# Tiếp tục Huấn luyện (Train Resume)

Notebook này giúp tiếp tục quá trình training từ các file checkpoint (`chkpnt15000.pth`) của lần chạy trước.

In [ ]:
# 1. Clone và cài đặt
!git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive
%cd gaussian-splatting
!pip install -q submodules/diff-gaussian-rasterization
!pip install -q submodules/simple-knn

In [ ]:
import os
import glob
import subprocess

# 2. Cấu hình đường dẫn và scene cần train tiếp
DATASET_ROOT = '/kaggle/input/converted-bts-dataset/private_set1'
CHECKPOINT_ROOT = '/kaggle/input/previous-training-outputs/outputs'
OUTPUT_ROOT = '/kaggle/working/outputs'

# Danh sách scene cần train. Để None hoặc [] nếu muốn resume TẤT CẢ
TARGET_SCENES = ['HCM0204']

TARGET_ITERATIONS = 30000  # Train tiếp lên 30k

if not os.path.exists(CHECKPOINT_ROOT):
    print(f"Thư mục checkpoint {CHECKPOINT_ROOT} không tồn tại.")
else:
    all_scenes = sorted(os.listdir(DATASET_ROOT))
    
    # Lọc scene theo TARGET_SCENES
    if TARGET_SCENES:
        scenes = [s for s in all_scenes if s in TARGET_SCENES]
    else:
        scenes = all_scenes
        
    for scene in scenes:
        scene_ckpt_dir = os.path.join(CHECKPOINT_ROOT, scene)
        
        # Tìm file checkpoint mới nhất (vd: chkpnt15000.pth)
        chkpnts = glob.glob(os.path.join(scene_ckpt_dir, "chkpnt*.pth"))
        if not chkpnts:
            print(f"Không tìm thấy checkpoint cho scene {scene}, bỏ qua!")
            continue
            
        latest_ckpt = max(chkpnts, key=os.path.getmtime)
        
        print("\n" + "="*60)
        print(f"RESUMING SCENE: {scene} FROM {os.path.basename(latest_ckpt)}")
        print("="*60)
        
        scene_path = os.path.join(DATASET_ROOT, scene, 'train')
        output_dir = os.path.join(OUTPUT_ROOT, scene)
        
        command = [
            "python", "train.py",
            "-s", scene_path,
            "-m", output_dir,
            "--eval",
            "--start_checkpoint", latest_ckpt,
            "--iterations", str(TARGET_ITERATIONS),
            "--test_iterations", "22000", str(TARGET_ITERATIONS),
            "--save_iterations", str(TARGET_ITERATIONS),
            "--checkpoint_iterations", str(TARGET_ITERATIONS),
            "--position_lr_max_steps", str(TARGET_ITERATIONS),
            "--disable_viewer"
        ]
        
        subprocess.run(command, check=True)
        print(f"Hoàn thành train 3DGS cho scene {scene}!")